In [ ]:
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix,balanced_accuracy_score
from sklearn.decomposition import PCA
import mlflow
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
import joblib

In [2]:
mlflow.set_experiment("binary_classification_experiment_v2")

2026/06/06 15:23:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/06 15:23:23 INFO mlflow.store.db.utils: Updating database tables
2026/06/06 15:23:24 INFO mlflow.tracking.fluent: Experiment with name 'binary_classification_experiment_v2' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:d:/fourth_year/graduation_project/JGuard/defenders/mult-iturn/NBF/notebooks/mlruns/1', creation_time=1780748604209, experiment_id='1', last_update_time=1780748604209, lifecycle_stage='active', name='binary_classification_experiment_v2', tags={}, trace_location=None, workspace='default'>

In [ ]:
def log_trained_model(model,y_pred,y_test,features,params: dict,run_name: str = "run",\
        model_name: str = "model",plot_name: str = "confusion_matrix.png",\
            dim_reduction_model=None):
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("features", str(features))

        bal_acc = balanced_accuracy_score(y_test, y_pred) # average of recall from each class
        f1_score_val = f1_score(y_test, y_pred, average='macro')
        classification_report_val = classification_report(y_test, y_pred, output_dict=True)

        mlflow.log_metric("balanced_accuracy", bal_acc)
        mlflow.log_metric("f1_score", f1_score_val)
        mlflow.log_dict(classification_report_val, "classification_report.json")
        
        cm = confusion_matrix(y_test, y_pred)
        mlflow.sklearn.log_model(model, model_name)
        if dim_reduction_model is not None:
            mlflow.sklearn.log_model(dim_reduction_model, f"{model_name}_dim_reduction")

        print(f"Run complete. Balanced Accuracy: {bal_acc}")
        return {"balanced_accuracy": bal_acc,"f1_score": f1_score_val,"confusion_matrix": cm.tolist()  }

In [2]:
data=torch.load("./../train_data_with_context/all_conversations.pt")
data2=torch.load("./../train_data_with_context/all_conversations2.pt")

In [3]:
data[0][0].keys()

dict_keys(['x_t', 'zt', 'score', 'y'])

In [4]:
labels_map={
    1:0,
    2:0,
    3:0,
    4:0,
    5:1
}

In [5]:
X = []
y=[]
for convo in data:
    for turn in convo:
        X.append(torch.concat([turn["x_t"], turn["zt"]], dim=0))
        y.append(labels_map[turn["score"]])

In [6]:
X2 = []
y2=[]
for convo in data2:
    for turn in convo:
        X2.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))
        y2.append(labels_map[turn["score"]])

In [7]:
from collections import Counter
c=Counter(y)
c

Counter({0: 26362, 1: 2617})

In [8]:
# high imbalance ratio
ratio=c[0]/c[1]
ratio

10.07336645013374

In [9]:
X = np.array(X)
y = np.array(y)
X2 = np.array(X2)
y2 = np.array(y2)

In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

In [11]:
# dimensionality reduction using PCA on the first dataset xt and zt features
pca1 = PCA(n_components=0.95, random_state=42)
X_train1_pca = pca1.fit_transform(X_train1)
X_test1_pca = pca1.transform(X_test1)
print("Number of components:", pca1.n_components_)
print("Total explained variance:", pca1.explained_variance_ratio_.sum())

# dimensionality reduction using PCA on the second dataset xt and ut features
pca2 = PCA(n_components=0.95, random_state=42)
X_train2_pca = pca2.fit_transform(X_train2)
X_test2_pca = pca2.transform(X_test2)
print("Number of components:", pca2.n_components_)
print("Total explained variance:", pca2.explained_variance_ratio_.sum())

Number of components: 213
Total explained variance: 0.95034
Number of components: 352
Total explained variance: 0.95001066


In [ ]:
# save the PCA models for later use
joblib.dump(pca1, "./../dim_reduction_models/mapping1_5/pca1_model.joblib")
joblib.dump(pca2, "./../dim_reduction_models/mapping1_5/pca2_model.joblib")

['./../dim_reduction_models/pca2_model.joblib']

In [ ]:
weights = {0: 1.0, 1: 8.0}  # kemeyet el loss 3ala kol class
svm_balanced = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight=weights)

y_train_pred = svm_balanced.fit(X_train1, y_train1).predict(X_train1)
y_test_pred = svm_balanced.predict(X_test1)

In [ ]:
print("train_f1_score:", f1_score(y_train1, y_train_pred, average='macro'))
print("test f1_score:", f1_score(y_test1, y_test_pred, average='macro'))
print(classification_report(y_test1, y_test_pred))

train_f1_score: 0.8348007990314081
Balanced f1_score: 0.7182060958637897
              precision    recall  f1-score   support

           0       0.97      0.90      0.93      5273
           1       0.40      0.68      0.50       523

    accuracy                           0.88      5796
   macro avg       0.68      0.79      0.72      5796
weighted avg       0.92      0.88      0.89      5796



In [17]:
log_trained_model(svm_balanced,y_test_pred,y_test1,features="Xt_Zt",\
                  params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"weights = '{0: 1.0, 1: 8.0}'"},\
                run_name="SVM_rbf_balanced",model_name="svm_rbf_different",plot_name="svm_rbf_balanced_cm.png")

2026/06/06 15:31:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:31:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:31:14 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmplu2tyebt\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7900


{'balanced_accuracy': 0.789993324338172,
 'f1_score': 0.7182060958637897,
 'confusion_matrix': [[4742, 531], [167, 356]]}

In [ ]:
# from sklearn.model_selection import GridSearchCV

# params = {
#     'C': [0.1, 1, 10],
#     'kernel': ['linear', 'rbf'],
#     'gamma': ['scale', 'auto']
# }

# grid = GridSearchCV(SVC(), params, cv=5)
# grid.fit(X_train1, y_train1)

# print("Best params:", grid.best_params_)

Best params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}


# with dimensionality reduction

In [18]:
weights = {0: 1.0, 1: 7.0}
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight=weights)  

y_train_pred=model.fit(X_train1_pca, y_train1).predict(X_train1_pca)
y_test_pred = model.predict(X_test1_pca)

In [19]:
print(classification_report(y_test1, y_test_pred))

              precision    recall  f1-score   support

           0       0.96      0.91      0.93      5273
           1       0.41      0.66      0.51       523

    accuracy                           0.88      5796
   macro avg       0.69      0.78      0.72      5796
weighted avg       0.91      0.88      0.90      5796



In [20]:
log_trained_model(run_name="SVM_pca_different_weights", model=model, y_pred=y_test_pred, y_test=y_test1, features="PCAXt_Zt", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"{0: 1.0, 1: 7.0}"},
                model_name="svm_rbf_different", plot_name="svm_rbf_different_cm.png", dim_reduction_model=pca1)

2026/06/06 15:32:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:32:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:32:37 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpt9_2rr33\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/06/06 15:32:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:32:37 WARNING mlflow.sklearn: Savi

Run complete. Balanced Accuracy: 0.7850


{'balanced_accuracy': 0.7849923797374627,
 'f1_score': 0.7218843294140127,
 'confusion_matrix': [[4780, 493], [176, 347]]}

# trial 2

In [ ]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced") 
y_train_pred =model.fit(X_train2, y_train2).predict(X_train2)
y_test_pred = model.predict(X_test2)

print("Accuracy:", accuracy_score(y_test2, y_test_pred))
print(classification_report(y_test2, y_test_pred))

Accuracy: 0.879399585921325
              precision    recall  f1-score   support

           0       0.97      0.90      0.93      5273
           1       0.40      0.68      0.50       523

    accuracy                           0.88      5796
   macro avg       0.68      0.79      0.72      5796
weighted avg       0.91      0.88      0.89      5796



In [22]:
log_trained_model(run_name="SVM_balanced_weights", model=model, y_pred=y_test_pred, y_test=y_test2, features="Xt_ut", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_balanced", plot_name="svm_rbf_balanced_cm.png")

2026/06/06 15:40:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:40:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:40:23 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpqfgoo_ft\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7899


{'balanced_accuracy': 0.7898985016565867,
 'f1_score': 0.7179746795614543,
 'confusion_matrix': [[4741, 532], [167, 356]]}

In [ ]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")  
model.fit(X_train2_pca, y_train2)
y_test_pred = model.predict(X_test2_pca)

print("Accuracy:", accuracy_score(y_test2, y_test_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test2, y_test_pred))
print("F1 Score:", f1_score(y_test2, y_test_pred, average='macro'))
print(classification_report(y_test2, y_test_pred))

Accuracy: 0.8814699792960663
Balanced Accuracy: 0.7832855714689249
F1 Score: 0.7176270057329805
              precision    recall  f1-score   support

           0       0.96      0.90      0.93      5273
           1       0.40      0.66      0.50       523

    accuracy                           0.88      5796
   macro avg       0.68      0.78      0.72      5796
weighted avg       0.91      0.88      0.89      5796



In [24]:
log_trained_model(run_name="SVM_balanced_weights_pca", model=model, y_pred=y_test_pred, y_test=y_test2, features="pca(Xt_ut)", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_balanced", plot_name="cm.png")

2026/06/06 15:41:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:41:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:41:56 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpxahh37nx\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7833


{'balanced_accuracy': 0.7832855714689249,
 'f1_score': 0.7176270057329805,
 'confusion_matrix': [[4762, 511], [176, 347]]}

# random forest

In [ ]:
rf = RandomForestClassifier(class_weight="balanced",random_state=42,n_jobs=-1)
param_dist = {
    "n_estimators": randint(200, 800),
    "max_depth": [None, 10, 20, 30, 40, 50],
    "min_samples_split": randint(2, 15),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy"]
}

random_search = RandomizedSearchCV(estimator=rf,
    param_distributions=param_dist,
    n_iter=30,           
    scoring="f1",         
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train1_pca, y_train1)
best_rf = random_search.best_estimator_
print("Best Parameters:")
print(random_search.best_params_)

y_test_pred = best_rf.predict(X_test1_pca)
print("\nF1 Score:", f1_score(y_test1, y_test_pred))
print(classification_report(y_test1, y_test_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'bootstrap': False, 'criterion': 'entropy', 'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 6, 'min_samples_split': 7, 'n_estimators': 305}

F1 Score: 0.4272179155900086
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      5273
           1       0.39      0.47      0.43       523

    accuracy                           0.89      5796
   macro avg       0.67      0.70      0.68      5796
weighted avg       0.90      0.89      0.89      5796

